# Google Colab Setup
Run this cell to install all required dependencies.


In [ ]:
!pip install transformers datasets evaluate captum scikit-learn pandas -q


# Adversarial Evaluation
Applying perturbations and checking Attack Success Rate (ASR) across all 5 architectures.


In [ ]:
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('../src'))
from models.lr_tfidf import LRTfidfModel
from models.roberta import RobertaModel
from models.bert import BertModel
from models.albert import AlbertModel
from evaluation import compute_attack_success_rate


In [ ]:
adv_df = pd.read_pickle('../data/processed/adv_test.pkl')
orig_true = adv_df['label'].tolist()


In [ ]:
models = {
    'lr_char': LRTfidfModel(model_dir='../models/lr_char', analyzer='char_wb'),
    'lr_word': LRTfidfModel(model_dir='../models/lr_word', analyzer='word'),
    'roberta': RobertaModel(model_dir='../models/roberta'),
    'bert': BertModel(model_dir='../models/bert'),
    'albert': AlbertModel(model_dir='../models/albert')
}


In [ ]:
for name, model in models.items():
    model.load()
    if 'lr' in name:
        orig_preds = model.predict(adv_df['original_text'].tolist())
        adv_preds = model.predict(adv_df['perturbed_text'].tolist())
    else:
        orig_preds, _ = model.predict(adv_df['original_text'].tolist(), batch_size=16)
        adv_preds, _ = model.predict(adv_df['perturbed_text'].tolist(), batch_size=16)
    asr = compute_attack_success_rate(orig_true, orig_preds, adv_preds)
    print(f'Overall Attack Success Rate ({name.upper()}): {asr:.2%}')
